Establishment of gut microbiome aging clocks (MicroAge)
To construct a gut microbiota‑based aging clock (MicroAge), elastic net regression was performed using the glmnet R package (version 4.1.8). For microbial compositional data, species‑level features were transformed using the centered log‑ratio (CLR) method. Sex and batch were incorporated into the model matrix as one‑hot encoded covariates. Samples were randomly split into training and testing sets at a 1:1 ratio. Within the training set, 10‑fold cross‑validation was applied to identify the optimal lambda (regularization parameter) for each candidate alpha (mixing parameter ranging from 0.05 to 0.95). The final model was selected based on the alpha and lambda combination that yielded the lowest mean absolute error (MAE) in the testing set. The performance of the aging clock was assessed by computing the correlation coefficient (R) and MAE between the predicted MicroAge and chronological age. All analyses were parallelized across multiple CPU cores to enhance computational efficiency. The optimized model was further applied to predict MicroAge in individuals from the validation cohort.

In [ ]:
rm(list=ls())
library(dplyr)
library(patchwork)
library(cluster)
library(clusterSim)
library(ade4)
library(caret)
library(pROC)
library(microbiome)
library(vegan)
library(ecodist)
library(compositions)
library(glmnet)
library(data.table)
library(hdi)
library(stabs)

######################################################################
#######  adding covar to feature matrix  by One-Hot Encoding   #######    
######################################################################

load_feature_matrix <- function(feature_matrix_filename,metadata_filename){
  
  df_input_metadata<-read.csv("E:/QZ/UniqueID_final_v3.csv",row.names = 1)%>%
    filter(Treat !="Used")%>%
    filter(Group =="Control")%>%
    mutate(covar1=ifelse(Sex=="F",1,0))%>%
    mutate(covar2=ifelse(Sex=="M",1,0))%>%
    mutate(covar3=ifelse(Batch == "Batch1",1,0))%>%
    mutate(covar4=ifelse(Batch == "Batch2",1,0))%>%
    mutate(covar5=ifelse(Batch == "Batch3",1,0))%>%
    dplyr::select(Metagenome,covar1,covar2,covar3,covar4,covar5)
  
  count_file_name="E:/QZ/01_metag/00_Rawdata/01_metaphlan/merged_abundance_table_read_count_species.txt"
  df_input_data = read.table(file             = count_file_name,
                             header           = TRUE,
                             sep              = "\t", 
                             row.names        = 1,
                             stringsAsFactors = FALSE,
                             check.names = F)
  df_input_data[1:5, 1:5]
  # df<-data.frame(t(df_input_data))
  df<-clr(t(df_input_data+1))
  rowSums(df)
  CLR_table<-data.frame(df)
  
  CLR_table$Metagenome<-rownames(CLR_table)
  tmp<-merge(CLR_table,df_input_metadata,by="Metagenome")
  rownames(tmp)<-tmp$Metagenome
  tmp<-tmp%>%
    dplyr::select(-Metagenome)
  microbes <- as.matrix(tmp)
  microbes[1:5, 1:5]
  return(microbes)
  # return(CLR_table)
}

theme1 <- theme(
  axis.title=element_text(size = 13), 
  legend.text=element_text(size=12),
  legend.title=element_text(size=12),
  axis.line = element_line(size=0.7), 
  # panel.border = element_blank(),
  panel.grid = element_blank(),
  plot.title = element_text(hjust = 0.5))

######################################################################
########              loading feature matrix                  ########    
######################################################################

metadata_filename="Input metadata"
feature_matrix_filename="Input matrix"

feature <- load_feature_matrix(feature_matrix_filename,metadata_filename)
dim(feature)

######################################################################
########                    loading metadata                  ########    
######################################################################

meta_data<-read.csv("E:/QZ/UniqueID_final_v3.csv",row.names = 1)%>%
  filter(Treat !="Used")%>%
  filter(Group =="Control")%>%
  dplyr::select(Age,Metagenome)

names(meta_data)

rownames(meta_data)  <- meta_data$Metagenome
meta_data<-meta_data%>%dplyr::select(Age)
table(meta_data$Age)
head(meta_data,5)

######################################################################
########                 filter feature (optional)            ########  
########                   Aging related feature              ########  
######################################################################

Prevalence_0.05_species_list<-read.csv("E:/QZ/01_metag/01_Prevalence_0.05/Prevalence_0.05_species_list.csv")
dim(feature)
feature=feature[,Prevalence_0.05_species_list$x]
dim(feature)

######################################################################
############  split feature matrix to train and test set  ############
######################################################################

sample_name=intersect(rownames(feature),rownames(meta_data))
sample_name

uniformed_ID_df_split_dataset_final<-read.csv("E:/QZ/UniqueID_final_v3.csv",row.names = 1)%>%
  filter(Treat !="Used")%>%
  filter(Group =="Control")

train_set<-uniformed_ID_df_split_dataset_final%>%filter(label=="train")
train_samples=train_set$Metagenome

test_set<-uniformed_ID_df_split_dataset_final%>%filter(label=="test")
test_samples=test_set$Metagenome

######################################################################
############  generate train and test set feature matrix  ############
######################################################################

Y_train_matrix <- as.matrix(meta_data[train_samples,])
Y_test_matrix <- as.matrix(meta_data[test_samples,])
X_train_matrix <- as.matrix(feature[train_samples,])
X_test_matrix<-as.matrix(feature[ test_samples,])


identical(rownames(X_train_matrix),train_samples)
identical(rownames(X_test_matrix),test_samples)

kfold = 10
x=X_train_matrix
y_i=Y_train_matrix
lambdas = NULL

result<-list()
n=1

dir.create("ElasticNet_result")

## glmnet CV
for (i in seq(0.05,0.95,0.05)){
  print(paste0("alpha ratio = ",i))
  dir.create(paste0("./ElasticNet_result/alpha_",i))
  
  cv.fit <- cv.glmnet(x, y_i, alpha=i, nfolds=kfold, type.measure = "mse", keep =TRUE, grouped=FALSE, standardize = T)  
  plot(cv.fit,label=T)
  
  ######################################################################
  ####################    get coef of model         ####################    
  ######################################################################
  saveRDS(cv.fit, paste0("./ElasticNet_result/alpha_",i,'/model.rds'))

  coef = summary(coef(cv.fit))
  coef$variable = c('Intercept', colnames(x)[coef$i-1])
  write.csv(coef, paste0("./ElasticNet_result/alpha_",i,'/model_coef.csv'))
  
  lambdas = data.frame(cv.fit$lambda,cv.fit$cvm)
  
  ## get best lambda -- lambda that gives min cvm
  bestlambda <- cv.fit$lambda.min
  bestlambda_index <- which(cv.fit$lambda == bestlambda)
  
  ## Get R^2 of final model
  final_model <- cv.fit$glmnet.fit
  
  actual = Y_train_matrix
  # a vector of integers for actual
  calculated = predict(cv.fit, newx = X_train_matrix, s = "lambda.min")
  nn = length(actual)
  sum = 0
  for (ii in 1:nn){sum = abs(actual[ii] - calculated[ii]) + sum}
  MAE_train = sum/nn
  
  
  actual = Y_test_matrix
  calculated = predict(cv.fit, newx = X_test_matrix, s = "lambda.min")
  nn = length(actual)
  sum = 0
  for (ii in 1:nn){sum = abs(actual[ii] - calculated[ii]) + sum}
  MAE_test = sum/nn
  
  ######################################################################
  ####################             model MAE        ####################    
  ######################################################################
  
  print("*****************")
  print(paste0("MAE in train set :",MAE_train," ! MAE in test set :",MAE_test," !"))
  print("*****************")
  
  result[[n]]<-data.frame(bestlambda = bestlambda,alpha=i,MAE_train=MAE_train,MAE_test=MAE_test)
  df<-result[[n]]
  n=n+1
  
  ######################################################################
  ########   generate the biological age of each individuals    ########   
  ######################################################################
  
  MAE_train=df$MAE_train
  MAE_test=df$MAE_test
  
  Finalbiologicalage=data.frame(predict(cv.fit, newx = X_train_matrix, s = "lambda.min"))
  
  colnames(Finalbiologicalage)="biologicalage"
  Finalbiologicalage$id=rownames(Finalbiologicalage)
  meta_data$id=rownames(meta_data)
  
  df<-merge(Finalbiologicalage,meta_data,by="id")
  
  train<-df
  train$Group="train"
  
  res<-cor.test(df$biologicalage,df$Age)
  # res$p.value
  # res$estimate
  
  df2 <- data.frame(
    x = c(30,30),
    y = c(70,80),
    text = c(paste0("MAE = ",round(MAE_train,3)),
             paste0("R = ",round(res$estimate[[1]],3))
             # paste0("P = ",round(res$p.value[[1]],3)))
    ))
  
  p1<-ggplot()+geom_point(data = df,aes(x=Age,y=biologicalage),color="black")+
    geom_smooth(data = df,aes(x=Age,y=biologicalage),method = "lm")+
    labs(y="Metab Age",x="Chronological age")+
    geom_text(data =df2,mapping = aes(x, y,label = text))+
    xlim(20,90)+ylim(20,90)+
    theme_light()+
    ggtitle("Train set")+theme1
  
  
  Finalbiologicalage=data.frame(predict(cv.fit, newx = X_test_matrix, s = "lambda.min"))
  
  colnames(Finalbiologicalage)="biologicalage"
  Finalbiologicalage$id=rownames(Finalbiologicalage)
  meta_data$id=rownames(meta_data)
  
  df<-merge(Finalbiologicalage,meta_data,by="id")
  test<-df
  test$Group="test"
  
  res<-cor.test(df$biologicalage,df$Age)
  # res$p.value
  # res$estimate
  
  df2 <- data.frame(
    x = c(30,30),
    y = c(70,80),
    text = c(paste0("MAE = ",round(MAE_test,3)),
             paste0("R = ",round(res$estimate[[1]],3))
             # paste0("P = ",round(res$p.value[[1]],3)))
    ))
  
  p2<-ggplot()+geom_point(data = df,aes(x=Age,y=biologicalage),color="black")+
    xlim(20,90)+ylim(20,90)+
    geom_smooth(data = df,aes(x=Age,y=biologicalage),method = "lm")+
    labs(y="Metab Age",x="Chronological age")+
    geom_text(data =df2,mapping = aes(x, y,label = text))+
    # ylim(0,max(aging$biologicalage)*1.2)+  
    theme_light()+
    ggtitle("Test set")+theme1
  p1+p2
  
  df<-rbind(train,test)
  write.csv(df,paste0("./ElasticNet_result/alpha_",i,'/BiologicalAge.csv'))
  ggsave(paste0("./ElasticNet_result/alpha_",i,'/Dotplot.pdf'),p1+p2,
         width =8, height =4,dpi=300)
  
}

df<-result[[1]]
for (i in seq(2,length(result))){
  df<-rbind(df,result[[i]])
}

write.csv(df,"ElasticNet_Alpha_Selection.csv")